In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Use TensorFlow‑Keras API (tf_keras provides a compatible tensorflow module)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

np.random.seed(42)
random.seed(42)



2026-05-05 16:46:50.509155: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777999610.522998      56 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777999610.527050      56 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 16:46:50.542560: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
base_path = "/kaggle/input/leaf-classification"
train_path = os.path.join(base_path, "train.csv")
test_path = os.path.join(base_path, "test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)



In [3]:
test_ids = test_df.pop("id")
train_ids = train_df.pop("id")  # kept for completeness
y_raw = train_df.pop("species")

le = LabelEncoder()
y_int = le.fit_transform(y_raw)

class_weights_array = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_int), y=y_int
)
class_weight_dict = dict(enumerate(class_weights_array))

scaler = StandardScaler()
X = scaler.fit_transform(train_df.values)  # (n_samples, n_features)
X_test = scaler.transform(test_df.values)  # (n_test, n_features)

y_cat = to_categorical(y_int)



In [4]:
num_features = X.shape[1]  # e.g., 192
num_classes = y_cat.shape[1]  # e.g., 99

model = Sequential()
model.add(
    Dense(
        512,
        input_shape=(num_features,),
        kernel_initializer="glorot_uniform",
        activation="relu",
    )
)
model.add(Dropout(0.3))
model.add(Dense(256, kernel_initializer="glorot_uniform", activation="relu"))
model.add(Dense(128, kernel_initializer="glorot_uniform", activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(64, kernel_initializer="glorot_uniform", activation="relu"))
model.add(Dense(num_classes, activation="softmax"))



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-05 16:46:57.538207: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777999617.538602      56 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


In [5]:
model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=5e-4),
    metrics=["accuracy"],
)

# Callbacks: save best validation model and reduce LR on plateau
checkpoint_path = "best_model.h5"
checkpoint_cb = ModelCheckpoint(
    checkpoint_path, monitor="val_loss", save_best_only=True, verbose=0
)
lr_cb = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=20, min_lr=1e-6, verbose=0
)

history = model.fit(
    X,
    y_cat,
    batch_size=32,
    epochs=800,  # more epochs to allow better convergence
    verbose=0,
    validation_split=0.1,
    shuffle=True,
    class_weight=class_weight_dict,
    callbacks=[checkpoint_cb, lr_cb],
)

# Load the best weights obtained on validation data
if os.path.exists(checkpoint_path):
    model.load_weights(checkpoint_path)



I0000 00:00:1777999619.530926     112 service.cc:148] XLA service 0x7ff54400bcd0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777999619.530959     112 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-05 16:46:59.555164: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777999619.669552     112 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-05 16:47:01.012738: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1147', 24 bytes spill stores, 24 bytes spill loads

2026-05-05 16:47:01.314897: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1147', 68 bytes spill 

In [6]:
y_pred = model.predict(X_test)  # softmax rows sum to 1
y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)  # safety clipping for log‑loss

species_cols = le.classes_
submission = pd.DataFrame(y_pred, columns=species_cols)
submission.insert(0, "id", test_ids.values)



4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 774ms/step


In [7]:
submission_path = "submission_nn_kernel.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission_nn_kernel.csv
